In [1]:
# ==========================================
# Real-time DD / McNichols-style OLS
# WITHOUT CFO_t+1
# WITH dREV
# SCALED BY AVERAGE TOTAL ASSETS: (AT_t + AT_t-1) / 2
# ==========================================

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path(".").resolve()

ACC_DIR = BASE_DIR / "acc_components_extracted"
PROF_DIR = BASE_DIR / "prof_components_extracted"

OUT_DIR = BASE_DIR / "acc_noise_ols_no_lead"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:

# -----------------------------
# 1. Load all accrual component CSVs
# -----------------------------
acc_files = sorted(ACC_DIR.glob("*.csv"))
print(f"Found {len(acc_files)} acc files in {ACC_DIR}")

acc_dfs = []
for fp in acc_files:
    try:
        df = pd.read_csv(fp)
        df["source_file_acc"] = fp.name
        acc_dfs.append(df)
    except Exception as e:
        print(f"Skipping acc file {fp.name}: {e}")

acc_panel = pd.concat(acc_dfs, ignore_index=True)
print(f"Combined acc panel shape: {acc_panel.shape}")


Found 633 acc files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/acc_components_extracted
Combined acc panel shape: (10354, 14)


In [3]:

# -----------------------------
# 2. Load REVT from prof files
# -----------------------------
prof_files = sorted(PROF_DIR.glob("*.csv"))
print(f"Found {len(prof_files)} prof files in {PROF_DIR}")

prof_dfs = []
for fp in prof_files:
    try:
        df = pd.read_csv(fp, usecols=["Year", "Ticker", "REVT"])
        df["source_file_prof"] = fp.name
        prof_dfs.append(df)
    except Exception as e:
        print(f"Skipping prof file {fp.name}: {e}")

prof_panel = pd.concat(prof_dfs, ignore_index=True)
print(f"Combined prof panel shape: {prof_panel.shape}")


Found 635 prof files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/prof_components_extracted
Combined prof panel shape: (10387, 4)


In [4]:

# -----------------------------
# 3. Basic cleaning
# -----------------------------
acc_panel["Year"] = pd.to_numeric(acc_panel["Year"], errors="coerce")
prof_panel["Year"] = pd.to_numeric(prof_panel["Year"], errors="coerce")

acc_panel = acc_panel.sort_values(["Ticker", "Year"]).reset_index(drop=True)
prof_panel = prof_panel.sort_values(["Ticker", "Year"]).reset_index(drop=True)

acc_num_cols = ["ACT", "CHE", "LCT", "STD", "TXP", "PPEGT", "AT", "OANCF"]
for c in acc_num_cols:
    acc_panel[c] = pd.to_numeric(acc_panel[c], errors="coerce")

prof_panel["REVT"] = pd.to_numeric(prof_panel["REVT"], errors="coerce")

# Assumption: missing STD and TXP are treated as zero
acc_panel["STD"] = acc_panel["STD"].fillna(0.0)
acc_panel["TXP"] = acc_panel["TXP"].fillna(0.0)


In [5]:

# -----------------------------
# 4. Merge REVT into accrual panel
# -----------------------------
panel = acc_panel.merge(
    prof_panel[["Ticker", "Year", "REVT"]],
    on=["Ticker", "Year"],
    how="left",
    validate="m:1"
)

print(f"Panel shape after merge: {panel.shape}")


Panel shape after merge: (10354, 15)


In [6]:

# -----------------------------
# 5. Construct DD / McNichols variables
# -----------------------------
g = panel.groupby("Ticker", group_keys=False)

# Lagged assets and average assets
panel["AT_lag1"] = g["AT"].shift(1)
panel["AvgAT"] = (panel["AT"] + panel["AT_lag1"]) / 2

# Balance sheet changes
panel["dACT"] = g["ACT"].diff()
panel["dCHE"] = g["CHE"].diff()
panel["dLCT"] = g["LCT"].diff()
panel["dSTD"] = g["STD"].diff()
panel["dTXP"] = g["TXP"].diff()
panel["dREV"] = g["REVT"].diff()

# Working capital accruals
panel["WCA"] = (panel["dACT"] - panel["dCHE"]) - (panel["dLCT"] - panel["dSTD"] - panel["dTXP"])

# Cash flow variables
panel["CFO_t-1"] = g["OANCF"].shift(1)
panel["CFO_t"] = panel["OANCF"]

# Scale by average total assets
panel["WCA_scaled"] = panel["WCA"] / panel["AvgAT"]
panel["CFO_t-1_scaled"] = panel["CFO_t-1"] / panel["AvgAT"]
panel["CFO_t_scaled"] = panel["CFO_t"] / panel["AvgAT"]
panel["dREV_scaled"] = panel["dREV"] / panel["AvgAT"]
panel["PPE_scaled"] = panel["PPEGT"] / panel["AvgAT"]


In [7]:

# -----------------------------
# 6. Keep regression sample
# -----------------------------
reg_cols = [
    "Ticker", "Year", "CompanyName", "Industry", "Sector",
    "WCA_scaled", "CFO_t-1_scaled", "CFO_t_scaled", "dREV_scaled", "PPE_scaled"
]

reg_df = panel[reg_cols].copy()
reg_df = reg_df.replace([np.inf, -np.inf], np.nan)

print("\nMissing values before regression drop:")
print(
    reg_df[["WCA_scaled", "CFO_t-1_scaled", "CFO_t_scaled", "dREV_scaled", "PPE_scaled"]]
    .isna()
    .sum()
)

reg_df = reg_df.dropna(
    subset=["WCA_scaled", "CFO_t-1_scaled", "CFO_t_scaled", "dREV_scaled", "PPE_scaled"]
).reset_index(drop=True)

print(f"\nRegression sample shape after dropna: {reg_df.shape}")
display(reg_df.head())



Missing values before regression drop:
WCA_scaled        798
CFO_t-1_scaled    733
CFO_t_scaled      701
dREV_scaled       660
PPE_scaled        913
dtype: int64

Regression sample shape after dropna: (9327, 10)


,Ticker,Year,CompanyName,Industry,Sector,WCA_scaled,CFO_t-1_scaled,CFO_t_scaled,dREV_scaled,PPE_scaled
0,20202.OL,2019,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,-0.008944,-0.002650,0.029143,0.052000,1.076466
1,20202.OL,2020,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,-0.002793,0.015263,0.069789,0.129308,1.086343
2,20202.OL,2021,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,0.001355,0.060039,0.210021,0.142753,0.929358
3,20202.OL,2022,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,0.022344,0.200447,0.113507,-0.060570,0.954941
4,20202.OL,2023,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,-0.024475,0.107903,0.115425,0.001142,0.907951


In [8]:

# -----------------------------
# 7. Cross-sectional OLS by year
# -----------------------------
yearly_results = []
residual_frames = []

min_obs_per_year = 20

for year, sub in reg_df.groupby("Year"):
    sub = sub.copy()

    if len(sub) < min_obs_per_year:
        print(f"Skipping {year}: only {len(sub)} observations")
        continue

    y = sub["WCA_scaled"]
    X = sub[["CFO_t-1_scaled", "CFO_t_scaled", "dREV_scaled", "PPE_scaled"]]
    X = sm.add_constant(X)

    try:
        model = sm.OLS(y, X).fit(cov_type="HC3")

        yearly_results.append({
            "Year": year,
            "n_obs": len(sub),
            "r2": model.rsquared,
            "adj_r2": model.rsquared_adj,
            "coef_const": model.params.get("const", np.nan),
            "coef_CFO_t-1": model.params.get("CFO_t-1_scaled", np.nan),
            "coef_CFO_t": model.params.get("CFO_t_scaled", np.nan),
            "coef_dREV": model.params.get("dREV_scaled", np.nan),
            "coef_PPE": model.params.get("PPE_scaled", np.nan),
            "t_CFO_t-1": model.tvalues.get("CFO_t-1_scaled", np.nan),
            "t_CFO_t": model.tvalues.get("CFO_t_scaled", np.nan),
            "t_dREV": model.tvalues.get("dREV_scaled", np.nan),
            "t_PPE": model.tvalues.get("PPE_scaled", np.nan),
            "p_CFO_t-1": model.pvalues.get("CFO_t-1_scaled", np.nan),
            "p_CFO_t": model.pvalues.get("CFO_t_scaled", np.nan),
            "p_dREV": model.pvalues.get("dREV_scaled", np.nan),
            "p_PPE": model.pvalues.get("PPE_scaled", np.nan),
        })

        sub["WCA_fitted"] = model.predict(X)
        sub["dd_resid"] = y - sub["WCA_fitted"]
        residual_frames.append(sub)

    except Exception as e:
        print(f"Skipping {year} due to regression error: {e}")

yearly_results_df = pd.DataFrame(yearly_results).sort_values("Year").reset_index(drop=True)
resid_df = pd.concat(residual_frames, ignore_index=True).sort_values(["Ticker", "Year"]).reset_index(drop=True)

print("\nYearly regression summary:")
display(yearly_results_df)

print("\nResidual sample preview:")
display(resid_df.head())


Skipping 1989: only 1 observations
Skipping 1990: only 1 observations
Skipping 1991: only 1 observations
Skipping 1992: only 1 observations
Skipping 1993: only 3 observations
Skipping 1994: only 3 observations
Skipping 1995: only 6 observations
Skipping 1996: only 9 observations
Skipping 1997: only 12 observations
Skipping 1998: only 15 observations

Yearly regression summary:


,Year,n_obs,r2,adj_r2,coef_const,coef_CFO_t-1,coef_CFO_t,coef_dREV,coef_PPE,t_CFO_t-1,t_CFO_t,t_dREV,t_PPE,p_CFO_t-1,p_CFO_t,p_dREV,p_PPE
0,1999,23,0.680043,0.608942,0.044418,0.322178,-0.672690,0.096203,-0.040291,1.091825,-2.030115,1.328110,-1.177713,0.274910,0.042345,0.184142,0.238911
1,2000,25,0.393634,0.272360,0.051387,-1.247487,1.512022,0.182822,-0.012120,-1.788761,1.670785,1.845256,-0.236645,0.073653,0.094764,0.065000,0.812932
2,2001,30,0.130558,-0.008553,-0.058514,0.366373,-0.245393,0.015576,0.023533,1.144129,-0.929734,0.338693,0.666045,0.252570,0.352509,0.734841,0.505383
3,2002,33,0.393225,0.306542,-0.088571,-0.067176,0.488782,0.120169,0.052957,-0.252652,1.286029,0.624063,1.332425,0.800537,0.198433,0.532586,0.182720
4,2003,35,0.142370,0.028019,0.044363,-0.345692,0.363615,-0.094362,-0.011825,-1.027226,0.974616,-0.708217,-0.543531,0.304314,0.329751,0.478811,0.586764
5,2004,53,0.169029,0.099781,0.071064,0.177577,-0.147219,0.079144,-0.109431,0.568216,-0.455516,1.030615,-2.131319,0.569888,0.648738,0.302722,0.033063
6,2005,251,0.016986,0.001002,0.010204,-0.158423,0.062652,0.010124,-0.000564,-1.310129,0.687112,0.220398,-0.057194,0.190152,0.492012,0.825561,0.954391
7,2006,292,0.117888,0.105594,-0.001639,0.250795,-0.037551,0.098213,0.026953,1.349032,-0.245310,1.277568,0.827486,0.177327,0.806216,0.201402,0.407962
8,2007,308,0.052878,0.040375,0.017629,0.160831,-0.206910,0.086916,-0.040713,1.282511,-1.846331,2.225298,-0.870226,0.199663,0.064844,0.026061,0.384177
9,2008,324,0.013902,0.001537,0.019002,0.012459,0.035212,0.023547,-0.015378,0.118983,0.301744,0.638627,-1.299430,0.905289,0.762847,0.523065,0.193796



Residual sample preview:


,Ticker,Year,CompanyName,Industry,Sector,WCA_scaled,CFO_t-1_scaled,CFO_t_scaled,dREV_scaled,PPE_scaled,WCA_fitted,dd_resid
0,20202.OL,2019,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,-0.008944,-0.002650,0.029143,0.052000,1.076466,-0.023078,0.014135
1,20202.OL,2020,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,-0.002793,0.015263,0.069789,0.129308,1.086343,-0.028003,0.025210
2,20202.OL,2021,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,0.001355,0.060039,0.210021,0.142753,0.929358,-0.018065,0.019420
3,20202.OL,2022,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,0.022344,0.200447,0.113507,-0.060570,0.954941,-0.001942,0.024286
4,20202.OL,2023,2020 Bulkers Ltd,Freight & Logistics Services,Industrials,-0.024475,0.107903,0.115425,0.001142,0.907951,-0.014805,-0.009671


In [9]:

# -----------------------------
# 8. Construct firm-year accounting noise
# Expanding std for first usable years,
# then 5-year rolling std
# -----------------------------
def expanding_then_rolling_std(x: pd.Series, rolling_window: int = 5, min_periods_start: int = 3) -> pd.Series:
    x = x.astype(float)
    out = []

    for i in range(len(x)):
        hist = x.iloc[:i+1].dropna()

        if len(hist) < min_periods_start:
            out.append(np.nan)
        elif len(hist) < rolling_window:
            out.append(hist.std(ddof=1))
        else:
            out.append(hist.iloc[-rolling_window:].std(ddof=1))

    return pd.Series(out, index=x.index)

resid_df["sigma_acc"] = (
    resid_df.groupby("Ticker", group_keys=False)["dd_resid"]
    .apply(expanding_then_rolling_std, rolling_window=5, min_periods_start=3)
)

# Optional robustness proxy: rolling mean absolute residual
def expanding_then_rolling_mean_abs(x: pd.Series, rolling_window: int = 5, min_periods_start: int = 3) -> pd.Series:
    x = x.astype(float)
    out = []

    for i in range(len(x)):
        hist = x.iloc[:i+1].dropna().abs()

        if len(hist) < min_periods_start:
            out.append(np.nan)
        elif len(hist) < rolling_window:
            out.append(hist.mean())
        else:
            out.append(hist.iloc[-rolling_window:].mean())

    return pd.Series(out, index=x.index)

resid_df["sigma_acc_abs"] = (
    resid_df.groupby("Ticker", group_keys=False)["dd_resid"]
    .apply(expanding_then_rolling_mean_abs, rolling_window=5, min_periods_start=3)
)


In [10]:

# -----------------------------
# 9. Diagnostics
# -----------------------------
print("\nCoverage of accounting-noise measures:")
print(resid_df[["sigma_acc", "sigma_acc_abs"]].notna().sum())

print("\nPreview with sigma:")
display(
    resid_df[
        ["Ticker", "Year", "WCA_scaled", "WCA_fitted", "dd_resid", "sigma_acc", "sigma_acc_abs"]
    ].head(15)
)



Coverage of accounting-noise measures:
sigma_acc        8019
sigma_acc_abs    8019
dtype: int64

Preview with sigma:


,Ticker,Year,WCA_scaled,WCA_fitted,dd_resid,sigma_acc,sigma_acc_abs
0,20202.OL,2019,-0.008944,-0.023078,0.014135,NaN,NaN
1,20202.OL,2020,-0.002793,-0.028003,0.025210,NaN,NaN
2,20202.OL,2021,0.001355,-0.018065,0.019420,0.005540,0.019588
3,20202.OL,2022,0.022344,-0.001942,0.024286,0.005097,0.020763
4,20202.OL,2023,-0.024475,-0.014805,-0.009671,0.014308,0.018544
5,20202.OL,2024,0.004496,0.002510,0.001986,0.015412,0.016114
6,8TRA.ST,2017,0.044641,0.014844,0.029798,NaN,NaN
7,8TRA.ST,2018,0.132670,0.013521,0.119149,NaN,NaN
8,8TRA.ST,2019,0.076971,-0.015923,0.092894,0.045924,0.080613
9,8TRA.ST,2020,-0.021133,-0.019209,-0.001924,0.055759,0.060941


In [11]:

# -----------------------------
# 10. Save outputs
# -----------------------------
yearly_results_df.to_csv(OUT_DIR / "yearly_ols_summary_no_lead_avgat.csv", index=False)
resid_df.to_csv(OUT_DIR / "firm_year_residuals_and_sigma_no_lead_avgat.csv", index=False)

print(f"\nSaved outputs to: {OUT_DIR}")


Saved outputs to: /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/acc_noise_ols_no_lead
